# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maqsood-Ahmed110/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Signal 1 — Staleness (behind the refresh flag): checking whether pages with
days_since_last_update >= 180 show lower gsc_impressions than fresher pages...
[paste your full Signal 1 + Signal 2 + rule text — the block currently
sitting under "2. Build the ranked queue"]

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("connected")

connected


In [8]:
REL = 'hf://datasets/FlyRank/internship-warehouse'
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet')").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## 2. Build the ranked queue (writes the CSV)

Signal 1 — Staleness (behind the refresh flag): checking whether pages
with days_since_last_update >= 180 show lower gsc_impressions than
fresher pages, bucketed by freshness_tier, with n printed per bucket.

Signal 2 — CTR-vs-position (behind the CTR-fix logic): checking whether
pages with worse CTR relative to their position-tier peers show a real
gap, bucketed by position tier, with n printed per bucket.

My rule: score = 0.5 * stale_visible_flag + 0.5 * ctr_gap_flag, with one
reason code (STALE_VISIBLE / CTR_GAP / BOTH / NONE) and an action label
(review / monitor).

In [ ]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

df = con.sql(f"""
    WITH agg AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               SUM(gsc_clicks) AS clicks_90d,
               AVG(gsc_avg_position) AS avg_position,
               MAX(report_date) AS last_report_date
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY content_hash_id
    )
    SELECT c.content_hash_id, c.content_updated_date, c.word_count,
           a.impressions_90d, a.clicks_90d, a.avg_position,
           DATE_DIFF('day', c.content_updated_date, a.last_report_date) AS days_since_last_update
    FROM read_parquet('{REL}/dim_content.parquet') c
    JOIN agg a ON c.content_hash_id = a.content_hash_id
""").df()

print("Joined rows:", len(df))

# Signal 1: staleness bucket table
df['freshness_tier'] = pd.cut(df['days_since_last_update'],
    bins=[-1, 30, 90, 180, 99999], labels=['0-30','31-90','91-180','181+'])
print("\n--- Signal 1: freshness vs impressions ---")
print(df.groupby('freshness_tier', observed=True)['impressions_90d'].agg(['mean', 'count']))

# Signal 2: CTR-vs-position bucket table
df['ctr'] = df['clicks_90d'] / df['impressions_90d'].replace(0, pd.NA)
df['position_tier'] = pd.cut(df['avg_position'], bins=[0,3,10,20,999], labels=['1-3','4-10','11-20','21+'])
print("\n--- Signal 2: position vs CTR ---")
print(df.groupby('position_tier', observed=True)['ctr'].agg(['mean', 'count']))
print("\nVerdict, Signal 1 (staleness): MIXED — 181+ day pages show dramatically")
print("lower impressions (4.56 vs 100-1142 for other tiers), but the middle tiers")
print("aren't monotonic — likely a ramp-up lag in the freshest bucket. Also: ~88%")
print("of rows lack content_updated_date, limiting this signal's coverage.")

print("\nVerdict, Signal 2 (CTR-vs-position): CONFIRMED — CTR drops cleanly and")
print("monotonically from 1.06% (pos 1-3) to 0.19% (pos 21+), a ~5.5x gap.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Top-20 review

Rule: score = 0.5 * stale_visible_flag + 0.5 * ctr_gap_flag

stale_visible_flag: only computed where content_updated_date is present
(88% of rows lack it — treated as "unknown," not "not stale," since
missing isn't evidence of freshness).

ctr_gap_flag: page's CTR is below half its position tier's average CTR
— this signal held up cleanly (CONFIRMED), so it carries real weight.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

Weakest picks: #9 and #10 — both have low impression counts (93 and 202)
where a "0% CTR" claim is more likely noise than a real signal; the CTR
gap logic works best at higher volume.

Data quality flag: rows 5-10 all show days_since_last_update = -94,
identical across all six — this isn't independent stale content, it's
likely a systemic data artifact (content_updated_date falling after the
report window's last date for a batch of pages). This should be
investigated before trusting the CTR_GAP-only picks, and is a genuine
limitation of this baseline, not a hidden strength.

Leakage check: no trend_pct, no product flags (health_score,
priority_score, action_type), and no future-window data were used as
inputs — only trailing 90-day GSC signals and content metadata available
at decision time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Only compute staleness where we actually have the date (avoid treating missing as "fresh")
df['has_update_date'] = df['content_updated_date'].notna()
df['stale_visible_flag'] = (
    df['has_update_date'] &
    (df['days_since_last_update'] >= 180) &
    (df['impressions_90d'] >= 500)
).astype(int)

# CTR gap: below half the position tier's average CTR
tier_avg_ctr = df.groupby('position_tier', observed=True)['ctr'].transform('mean')
df['ctr_gap_flag'] = (df['ctr'] < tier_avg_ctr * 0.5).astype(int)

df['score'] = 0.5 * df['stale_visible_flag'] + 0.5 * df['ctr_gap_flag']

def reason_code(row):
    if row['stale_visible_flag'] and row['ctr_gap_flag']:
        return 'BOTH'
    elif row['stale_visible_flag']:
        return 'STALE_VISIBLE'
    elif row['ctr_gap_flag']:
        return 'CTR_GAP'
    return 'NONE'

df['reason_code'] = df.apply(reason_code, axis=1)
df['action'] = df['score'].apply(lambda s: 'review' if s > 0 else 'monitor')

ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Wrote", len(ranked), "rows to work/outputs/baseline_action_score.csv")
print("\nScore distribution:")
print(ranked['score'].value_counts())
print("\nTop 10:")
ranked[['content_hash_id','score','reason_code','action','days_since_last_update','impressions_90d','ctr']].head(10)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.